# Graze Smasher Product Hunting Tool
## Prototype Assignment: ML Based Fake Review Detection System
This notebook demonstrates the complete machine learning pipeline for detecting fake product reviews. It covers dataset collection, text preprocessing, model training, evaluation, and serialization.

## Task 1: Dataset Collection
We collect the **Salminen et al. Fake Reviews Dataset** containing balanced original (OR) and computer-generated (CG) reviews. Here we load the dataset, print its dimensions, class distribution, and display the first five records.

In [ ]:
import pandas as pd
import os

# Load the dataset
dataset_path = r'../Dataset/fake_reviews_dataset.csv'
df = pd.read_csv(dataset_path)

# Display dataset overview
print('Dataset Dimensions:', df.shape)
print('\nClass Distribution:')
print(df['label'].value_counts())

# Display the first five records
df.head()

## Task 2: Data Preprocessing
We perform the following steps to clean and standardize the review text:
1. **Lowercase Conversion**: Convert all text to lowercase to ensure consistency.
2. **Punctuation Removal**: Remove standard punctuation marks which do not add semantic value.
3. **Tokenization**: Segment the cleaned text into individual word tokens.
4. **Stop-word Removal**: Filter out high-frequency words (e.g. 'the', 'is', 'in') using NLTK's English stop-word list.
5. **Lemmatization**: Reduce words to their dictionary root form (using WordNetLemmatizer) for feature normalization.

Below we import the preprocessing pipeline and showcase original vs. preprocessed text.

In [ ]:
import sys
sys.path.append('../ML_Model')
from preprocess import preprocess_text

# Select a sample review from the dataset
sample_idx = 4
original_review = df.loc[sample_idx, 'text_']
processed_review = preprocess_text(original_review)

print('=== ORIGINAL REVIEW ===')
print(original_review)
print('\n=== PROCESSED REVIEW ===')
print(processed_review)

## Task 3: Machine Learning Model
We train a **Support Vector Machine (SVM)** classifier to predict whether a review is Fake or Genuine. The steps involved are:
1. Splitting the preprocessed data into an 80% training set and a 20% testing set.
2. Fitting a **TF-IDF Vectorizer** (Term Frequency-Inverse Document Frequency) on the training set to convert text into numeric vectors.
3. Fitting the `LinearSVC` model.
4. Evaluating performance on the testing set and printing: **Accuracy**, **Precision**, **Recall**, and **F1-Score**.
5. Exporting the trained model and vectorizer as `.pkl` files.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
import joblib

# Prepare text and labels
df['clean_text'] = df['text_'].fillna('').apply(preprocess_text)
df['label_mapped'] = df['label'].map({'CG': 'Fake', 'OR': 'Genuine'})

X = df['clean_text']
y = df['label_mapped']

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

# TF-IDF Vectorizer
vectorizer = TfidfVectorizer(min_df=2, sublinear_tf=True, ngram_range=(1, 2))
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Train SVM Classifier
svm_model = LinearSVC(C=1.0, random_state=42, dual=False)
svm_model.fit(X_train_vec, y_train)

# Predict on Test Set
y_pred = svm_model.predict(X_test_vec)

### Model Metrics & Evaluation Report

In [ ]:
# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted')

print('='*40)
print('             MODEL METRICS')
print('='*40)
print(f'Accuracy:  {accuracy*100:.2f}%')
print(f'Precision: {precision*100:.2f}%')
print(f'Recall:    {recall*100:.2f}%')
print(f'F1-Score:  {f1*100:.2f}%')
print('='*40)

print('\nClassification Report:')
print(classification_report(y_test, y_pred))

print('Confusion Matrix:')
cm = confusion_matrix(y_test, y_pred, labels=['Genuine', 'Fake'])
print(pd.DataFrame(cm, index=['Actual Genuine', 'Actual Fake'], columns=['Predicted Genuine', 'Predicted Fake']))

### Save the Model & Vectorizer

In [ ]:
# Save to Trained_Model/
os.makedirs('../Trained_Model', exist_ok=True)
joblib.dump(vectorizer, '../Trained_Model/vectorizer.pkl')
joblib.dump(svm_model, '../Trained_Model/svm_model.pkl')
print('Saved vectorizer and model to ../Trained_Model/')

## Model Demonstration (Loaded Model Predictions)
Below we demonstrate how to load the serialized model files and run a prediction on a new user review.

In [ ]:
# Load files
loaded_vectorizer = joblib.load('../Trained_Model/vectorizer.pkl')
loaded_model = joblib.load('../Trained_Model/svm_model.pkl')

def predict_new_review(text):
    cleaned = preprocess_text(text)
    vec = loaded_vectorizer.transform([cleaned])
    prediction = loaded_model.predict(vec)[0]
    return prediction

# Test case 1
test_review_1 = 'I absolutely love this keyboard! The key travel is perfect and the RGB looks clean.'
print(f'Review: "{test_review_1}"')
print('Prediction:', predict_new_review(test_review_1))

# Test case 2
test_review_2 = 'BUY CHEAP PHONES NOW!!! GO TO SPAM LINK AND RECEIVE FREE OFFERS IMMEDIATELY!!!'
print(f'\nReview: "{test_review_2}"')
print('Prediction:', predict_new_review(test_review_2))